# Study correlation length-scale of atmospheric seeing

Some resources

- Piérre-François wrote some code inside this folder:  
  `/sdf/data/rubin/user/leget/lsst_dev/tickets/PFMeters/20251114`.

Associated tickets:
* [RSO-36](https://rubinobs.atlassian.net/browse/RSO-36)

In [ ]:
import glob
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import os
import pandas as pd
import pickle

from astropy.table import Table
from scipy.stats import binned_statistic_2d
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from tqdm import tqdm  # just a simple progress bar

## Reduced data

We already have some processed data living in `/sdf/data/rubin/user/leget/lsst_dev/tickets/PFMeters/visitPkls`.  
Instead of running the analysis from zero, let me try to play with these pickle files and see if I can extract anything useful.

In [ ]:
def convert_dict_with_tables_in_dataframe(my_dict : dict) -> pd.DataFrame:
    """
    Each item in the dictionary returned by the pickle file is an AstroPy Table,
    which can be easily converted into a Pandas DataFrame using a simple method.
    """
    df = pd.DataFrame()
    
    for key, val in my_dict.items():
 
        if val is None:
            continue
           
        sub_df = val.to_pandas()
        df = pd.concat([df, sub_df], ignore_index=True)

    return df

### Single-Visit Analysis

Before doing anything, let me see if I can read the data and manipulate it a bit. 

In [ ]:
# Folder containing data processed by Pierre-François
PICKLE_FOLDER : str = "/sdf/data/rubin/user/leget/lsst_dev/tickets/PFMeters/visitPkls"

# Read all the pickle files inside that folder
list_of_filenames : list = [f for f in glob.glob(os.path.join(PICKLE_FOLDER, "*.pkl"))]

# Let's pay a bit with the first file. I will modify this code later.
filename : str = list_of_filenames[1]
print(f"Let's start our analysis with the file: {filename}")

# Let's open this pickle file
print(" Loading data...")
pickle_data : dict = pickle.load(open(filename, "rb"))
print(" Done!")

# I find it easier to work with Pandas DataFrames.
print(" Convert dict[table] into a dataframe.")
df : pd.DataFrame = convert_dict_with_tables_in_dataframe(pickle_data)
print(" Done!")

In [ ]:
# Grab filename from the original input
root_name, _ = os.path.splitext(os.path.split(filename)[-1])

# Let's pickup a column (we have many) to display
col = "T_src"

# Let's now calculate the minimum and maximum value based on the data itself
data = df[col]
vmin = np.nanmedian(data) - 2 * np.nanstd(data)
vmax = np.nanmedian(data) + 2 * np.nanstd(data)

cmap = mpl.cm.Spectral
bounds = np.round(np.linspace(vmin, vmax, 9), 1)
norm = mpl.colors.BoundaryNorm(bounds, cmap.N, extend='both')

# Now let's prepare our figure to plot
fig, ax = plt.subplots(num=root_name, figsize=(12, 7), dpi=96)

sct = ax.scatter(df["xFoV"], df["yFoV"], c=data, s=1, cmap=cmap, norm=norm)

cbar = ax.figure.colorbar(sct, ax=ax)
cbar.ax.set_ylabel(col, rotation=90)

# Add some cosmetics
ax.set_aspect('equal', adjustable='box') 
ax.set_xlabel('x (mm)')
ax.set_ylabel('y (mm)')

fig.tight_layout()
fig.suptitle(root_name)
fig.savefig(f"plots/{root_name}.png")

plt.show()

<br><br>  
Our data is sparse. We can fill the gaps using `binned_statistic_2d`. 

In [ ]:
# Use `binned_statistic_2d` to fill up gaps with missing data
stats, x_edges, y_edges, bin_number = binned_statistic_2d(x=df["xFoV"], y=df["yFoV"], values=data, bins=400)

# We need to rebuild a grid using the bin edges returned from the function above
x_center = 0.5 * (x_edges[1:] + x_edges[:-1])
y_center = 0.5 * (y_edges[1:] + y_edges[:-1])
Y, X = np.meshgrid(y_center, x_center)

# And now we can plot
fig, ax = plt.subplots(num=f"binned - {root_name}", figsize=(15, 7), dpi=96)

sct = ax.scatter(X, Y, c=stats, s=5, cmap=cmap, norm=norm)

cbar = ax.figure.colorbar(sct, ax=ax)
cbar.ax.set_ylabel(col, rotation=90)

# Add some cosmetics
ax.set_aspect('equal', adjustable='box') 
ax.set_xlabel('x (mm)')
ax.set_ylabel('y (mm)')

fig.suptitle(root_name)
fig.tight_layout()
fig.savefig(f"plots/stats_{root_name}.png")

plt.show()

### Multi-Visit Analysis

Right. Now that I have a better understanding of the data itself, let's do some patch processing.  
I will read each file, convert the dictionary containing tables into a pandas dataframe,  
extract the `T_src` column (what does it mean?),  
calculate the binned 2d statistics to augment the data,  
reshape the array so it is a single dimension array,
append the result to another array.  

The final result must be a 2D array where each row corresponds to the data associated to a single visit.  
The columns are the binned `T_src` squeezed in a single dimention.

In [ ]:
# Folder containing data processed by Pierre-François
PICKLE_FOLDER : str = "/sdf/data/rubin/user/leget/lsst_dev/tickets/PFMeters/visitPkls"

# Read all the pickle files inside that folder
list_of_filenames : list = [f for f in glob.glob(os.path.join(PICKLE_FOLDER, "*.pkl"))]

# Let's work with a single column for now
col = "T_src"

# Let's use n_bins in both X and Y
n_bins = 200

# Stacked data - contains data from all the visits
stacked_data = []

# Loop over each file
for pickle_fname in tqdm(list_of_filenames):
  
    _pickle_data : dict = pickle.load(open(pickle_fname, "rb"))
    _df : pd.DataFrame = convert_dict_with_tables_in_dataframe(_pickle_data)

    if _df.index.size == 0:
        continue
    
    _x : pd.Series = _df["xFoV"]
    _y : pd.Series = _df["yFoV"]
    _data : pd.Series = _df[col]
    
    _stats, _, _, _ = binned_statistic_2d(x=_x, y=_y, values=_data, bins=n_bins)
    _stats = _stats.ravel() # "Unroll" the data and make it 1D.

    stacked_data.append(_stats)


# Convert everything back into a 2D array. 
# Each row corresponds to the data of a single visit. 
# Each column corresponds to the binned data into a particular bin.
stacked_data = np.array(stacked_data)

<br><br>
Our data has lots of NaNs and gaps.  
Use the code below to fill up the gaps. 

In [ ]:
# Step 1: Find and remove all-NaN columns
all_nan_cols = np.all(np.isnan(stacked_data), axis=0)
nan_col_indices = np.where(all_nan_cols)[0]
np.save('nan_columns.npy', nan_col_indices)  # Save for later

# Step 2: Keep only columns with at least some data
data_clean = stacked_data[:, ~all_nan_cols]

# Step 3: Now impute the remaining NaNs
imputer = SimpleImputer(strategy='mean')
data_imputed = imputer.fit_transform(data_clean)

# data_imputed = data_clean.copy()
# data_imputed[np.isnan(data_imputed)] = np.nanmean(data_imputed)

<br><br>
Now that we have our data in a specific format, let's try to apply PCA to it.

In [ ]:
# Step 4: Apply PCA
pca = PCA(n_components=175)
pca.fit(data_imputed)

<br><br>
Review the PCA with some metrics and plots.

In [ ]:
# 1. Explained variance per component
print(pca.explained_variance_ratio_)

# 2. Cumulative explained variance
cumsum = np.cumsum(pca.explained_variance_ratio_)
print(f"50 components explain {cumsum[-1]:.1%} of variance")

# 3. Scree plot (see elbow)
plt.semilogy(pca.explained_variance_ratio_)
plt.xlabel('Component')
plt.ylabel('Explained Variance Ratio')
plt.show()

# 4. Cumulative variance plot
plt.plot(cumsum)
plt.axhline(y=0.95, color='r', linestyle='--')  # 95% threshold
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.show()

# 5. Reconstruction error (if needed)
reconstructed = pca.inverse_transform(pca.transform(data_imputed))
mse = np.mean((data_imputed - reconstructed)**2)

In [ ]:
# Reshape components back to your grid shape
n_bins_x, n_bins_y = n_bins, n_bins  # or whatever your grid size is

fig, axes = plt.subplots(10, 2, figsize=(15, 30), num="pca_analysis")

for i, ax in enumerate(axes.flat):
    
    # Reconstruct the component with NaN columns added back
    component_full = np.zeros(stacked_data.shape[1])
    component_full[~all_nan_cols] = pca.components_[i]
    component_full[all_nan_cols] = np.nan

    _vmin = np.nanmedian(component_full) - 2 * np.nanstd(component_full)
    _vmax = np.nanmedian(component_full) + 2 * np.nanstd(component_full)

    _cmap = mpl.cm.Spectral.copy()  # Copy the colormap
    _cmap.set_bad(color='white')  # Set color for NaN/masked values
    _bounds = np.round(np.linspace(_vmin, _vmax, 9), 1)
    _norm = mpl.colors.BoundaryNorm(_bounds, _cmap.N, extend='both')
    
    # Reshape to spatial grid
    pc_map = component_full.reshape(n_bins_y, n_bins_x)
    pc_map = np.ma.masked_invalid(pc_map)
    pc_map = pc_map[::-1, ::-1] # Get the orientation right
    
    im = ax.imshow(pc_map, cmap=_cmap, vmin=_vmin, vmax=_vmax, origin="lower")
    
    cbar = ax.figure.colorbar(im, ax=ax)
    cbar.ax.set_ylabel(col, rotation=90)

    ax.set_title(f'PC{i+1} ({pca.explained_variance_ratio_[i]:.1%})')  
    
    
fig.tight_layout()

root_fname, _ = os.path.splitext(os.path.split(pickle_fname)[-1])
fig.savefig(f"plots/pca_{root_fname[:8]}.png")